# Building a Splink model

Chapters 2.4 and 2.5 built a probabilistic linkage model by hand and found its
limit: with **exact** comparison, recall stalls around 0.25, because genuine
matches agree exactly on a given name only a third of the time.

This notebook rebuilds the same model in [Splink](https://moj-analytical-services.github.io/splink/),
and adds the thing that was missing — comparisons that recognise values as
*similar* rather than requiring them to be identical.

**What you will do**

1. Set up the data the way Splink needs it
2. Express the blocking rules from chapter 2.5 as Splink rules, and count them
   before committing
3. Choose comparison functions, and look at the levels they generate
4. Assemble the settings and build a `Linker`
5. Run a first prediction with untrained parameters, and see why it is not to be
   trusted yet

## 0. Setup and prepared data

The same preparation as chapters 2.2 to 2.5, plus the phonetic codes that both
the blocking rules and the comparisons will use.

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 130)

DATA = Path("../../data")
if not DATA.exists():
    DATA = Path("data")

fonasa = pd.read_csv(DATA / "fonasa_sample.csv", dtype=str)
suseso = pd.read_csv(DATA / "suseso_sample.csv", dtype=str)


def basic_text_clean(series):
    return (series.astype("string").str.strip().str.upper()
            .str.replace(r"\s+", " ", regex=True))


def remove_accents(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(c for c in value if not unicodedata.combining(c))


def standardise_name(series):
    cleaned = basic_text_clean(series)
    cleaned = cleaned.map(remove_accents, na_action="ignore").astype("string")
    cleaned = cleaned.str.replace(r"[^A-ZN ]", "", regex=True)
    return cleaned.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_sex(series):
    cleaned = basic_text_clean(series)
    return cleaned.replace({"HOMBRE": "M", "MUJER": "F", "MASCULINO": "M",
                            "FEMENINO": "F", "": pd.NA})


for df in (fonasa, suseso):
    for col in ["nombre", "ap1", "ap2"]:
        df[f"{col}_clean"] = standardise_name(df[col])
    df["sexo_clean"] = clean_sex(df["sexo"])
    df["nac_clean"] = basic_text_clean(df["nacionalidad"])

CLEAN = ["nombre_clean", "ap1_clean", "ap2_clean", "sexo_clean", "nac_clean"]
TRUE_MATCHES = 4500

print(f"fonasa: {len(fonasa):,} records | suseso: {len(suseso):,} records")
print(f"true matches present in the data: {TRUE_MATCHES:,}")


import phonetics


def dmeta_primary(value):
    """Primary Double Metaphone code, as a string. Used for blocking."""
    if pd.isna(value) or value == "":
        return None
    try:
        return phonetics.dmetaphone(str(value))[0] or None
    except Exception:
        return None


def dmeta_list(value):
    """Both Double Metaphone codes, as a list. This is the form Splink's
    comparison functions expect for a `dmeta_col_name` argument."""
    if pd.isna(value) or value == "":
        return None
    try:
        return [c for c in phonetics.dmetaphone(str(value)) if c] or None
    except Exception:
        return None


for df in (fonasa, suseso):
    for col in ["nombre_clean", "ap1_clean", "ap2_clean"]:
        stem = col.replace("_clean", "")
        df[f"{stem}_dm"] = df[col].map(dmeta_primary)      # string, for blocking
        df[f"{stem}_dmeta"] = df[col].map(dmeta_list)      # list, for comparisons

MODEL_COLS = (["unique_id"] + CLEAN
              + ["nombre_dm", "ap1_dm", "ap2_dm"]
              + ["nombre_dmeta", "ap1_dmeta", "ap2_dmeta"])
fonasa_m = fonasa[MODEL_COLS + ["true_person_id"]].copy()
suseso_m = suseso[MODEL_COLS + ["true_person_id"]].copy()

print("phonetic example:  GONZALEZ ->",
      dmeta_primary("GONZALEZ"), "(blocking) /", dmeta_list("GONZALEZ"), "(comparison)")

fonasa: 30,000 records | suseso: 27,000 records
true matches present in the data: 4,500
phonetic example:  GONZALEZ -> KNSLS (blocking) / ['KNSLS'] (comparison)


## 1. Why a library

Everything so far was pandas, and it worked. Three things make that
impractical past this point.

**Scale.** Splink compiles its work to SQL and executes it in an analytical
database engine — DuckDB here, and Spark or Athena at national scale. The same
model definition runs on a laptop and on a cluster.

**Estimating *m* without the answer key.** In chapter 2.4 we computed *m* from
4,500 known matches. You will not have those. Splink estimates *m* from the data
itself, by expectation-maximisation, which is not something you want to
implement yourself.

**Comparisons with levels.** A hand-rolled comparison is binary. Splink's
comparisons have several ordered levels — exact, then close, then loosely
similar, then everything else — and estimates a separate weight for each.

That last point is the one that matters most here, and it is what will finally
move recall.

## 2. Blocking rules

Splink's `block_on()` builds a rule from column names or SQL expressions. A
list of rules is combined as a **union**: a pair is a candidate if any rule
keeps it, which is exactly the disjunctive design of chapter 2.5.

These are the six rules that chapter's analysis settled on.

In [2]:
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl

blocking_rules = [
    block_on("nombre_clean", "ap1_clean"),
    block_on("ap1_clean", "ap2_clean"),
    block_on("substr(nombre_clean,1,1)", "ap1_clean", "ap2_clean"),
    block_on("ap1_dm", "ap2_dm"),
    block_on("nombre_dm", "ap1_dm"),
    block_on("substr(nombre_clean,1,1)", "ap1_dm", "ap2_dm"),
]

for r in blocking_rules:
    print(r.get_blocking_rule("duckdb").blocking_rule_sql)

(l."nombre_clean" = r."nombre_clean") AND (l."ap1_clean" = r."ap1_clean")
(l."ap1_clean" = r."ap1_clean") AND (l."ap2_clean" = r."ap2_clean")
(SUBSTRING(l.nombre_clean, 1, 1) = SUBSTRING(r.nombre_clean, 1, 1)) AND (l."ap1_clean" = r."ap1_clean") AND (l."ap2_clean" = r."ap2_clean")
(l."ap1_dm" = r."ap1_dm") AND (l."ap2_dm" = r."ap2_dm")
(l."nombre_dm" = r."nombre_dm") AND (l."ap1_dm" = r."ap1_dm")
(SUBSTRING(l.nombre_clean, 1, 1) = SUBSTRING(r.nombre_clean, 1, 1)) AND (l."ap1_dm" = r."ap1_dm") AND (l."ap2_dm" = r."ap2_dm")


Each rule becomes a SQL join condition. Nothing is hidden: you can read exactly
what the rule will do, which is one of the reasons blocking decisions are easy
to document.

**Count before you commit.** A rule that generates a billion pairs is not a rule,
and you want to know that before waiting for it rather than after.

In [3]:
from splink.blocking_analysis import count_comparisons_from_blocking_rule

db_api = DuckDBAPI()

rows = []
for rule in blocking_rules:
    counts = count_comparisons_from_blocking_rule(
        table_or_tables=[fonasa_m, suseso_m],
        blocking_rule=rule,
        link_type="link_only",
        db_api=db_api,
    )
    rows.append({
        "rule": rule.get_blocking_rule("duckdb").blocking_rule_sql[:58],
        "pairs": counts["number_of_comparisons_to_be_scored_post_filter_conditions"],
    })

counts_df = pd.DataFrame(rows)
counts_df["pct_of_all_pairs"] = (
    100 * counts_df["pairs"] / (len(fonasa_m) * len(suseso_m))
).round(6)
counts_df

,rule,pairs,pct_of_all_pairs
0,"(l.""nombre_clean"" = r.""nombre_clean"") AND (l.""...",1291,0.000159
1,"(l.""ap1_clean"" = r.""ap1_clean"") AND (l.""ap2_cl...",3546,0.000438
2,"(SUBSTRING(l.nombre_clean, 1, 1) = SUBSTRING(r...",1338,0.000165
3,"(l.""ap1_dm"" = r.""ap1_dm"") AND (l.""ap2_dm"" = r....",7998,0.000987
4,"(l.""nombre_dm"" = r.""nombre_dm"") AND (l.""ap1_dm...",2070,0.000256
5,"(SUBSTRING(l.nombre_clean, 1, 1) = SUBSTRING(r...",1882,0.000232


Every rule is a rounding error against the 810 million pairs a full comparison
would need. The rules that use phonetic codes generate the most, which is the
price of the extra completeness chapter 2.5 measured.

Splink can also show how the union accumulates, rule by rule — useful for
spotting a rule that adds a great deal of work and very few new pairs.

In [4]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_data,
)

cumulative = cumulative_comparisons_to_be_scored_from_blocking_rules_data(
    table_or_tables=[fonasa_m, suseso_m],
    blocking_rules=blocking_rules,
    link_type="link_only",
    db_api=db_api,
)
cumulative

,blocking_rule,row_count,cumulative_rows,cartesian,match_key,start
0,"(l.""nombre_clean"" = r.""nombre_clean"") AND (l.""...",1291,1291,810000000,0,0
1,"(l.""ap1_clean"" = r.""ap1_clean"") AND (l.""ap2_cl...",2458,3749,810000000,1,1291
2,"(SUBSTRING(l.nombre_clean, 1, 1) = SUBSTRING(r...",0,3749,810000000,2,3749
3,"(l.""ap1_dm"" = r.""ap1_dm"") AND (l.""ap2_dm"" = r....",4458,8207,810000000,3,3749
4,"(l.""nombre_dm"" = r.""nombre_dm"") AND (l.""ap1_dm...",661,8868,810000000,4,8207
5,"(SUBSTRING(l.nombre_clean, 1, 1) = SUBSTRING(r...",0,8868,810000000,5,8868


Read the `row_count` column: it is the number of pairs each rule contributes
that earlier rules had not already found.

Two of the six rules contribute **zero**. Both are the ones built on the
given-name initial: by the time they run, the rules on full names and phonetic
codes have already captured everything they would have found. On this data they
are pure cost with no benefit, and could be dropped without changing the
candidate set at all.

That is exactly what this table is for. A rule that looked sensible in the
abstract turns out to be redundant given the others, and you can only see it by
counting. They are kept here so the totals match chapter 2.5, but on your own
data, drop what earns nothing.

The union comes to **8,868 candidate pairs** — the same figure chapter 2.5
arrived at with the same rules in pandas.

## 3. Comparisons

A **comparison** tells Splink how to compare one field. Splink's comparison
library provides ready-made ones with sensible levels.

The important shift from chapter 2.4 is that a comparison is no longer
agree/disagree. Look at what `NameComparison` actually generates.

In [5]:
example = cl.NameComparison("nombre_clean").get_comparison("duckdb")

for level in example.comparison_levels:
    print(f"  {str(level.label_for_charts):<48} {level.sql_condition[:60]}")

  nombre_clean is NULL                             "nombre_clean_l" IS NULL OR "nombre_clean_r" IS NULL
  Exact match on nombre_clean                      "nombre_clean_l" = "nombre_clean_r"
  Jaro-Winkler distance of nombre_clean >= 0.92    jaro_winkler_similarity("nombre_clean_l", "nombre_clean_r") 
  Jaro-Winkler distance of nombre_clean >= 0.88    jaro_winkler_similarity("nombre_clean_l", "nombre_clean_r") 
  Jaro-Winkler distance of nombre_clean >= 0.7     jaro_winkler_similarity("nombre_clean_l", "nombre_clean_r") 
  All other comparisons                            ELSE


Six ordered levels, from an exact match down through three degrees of
Jaro-Winkler similarity to "everything else". Splink estimates a separate *m*
and *u* — and therefore a separate weight — for **each** level.

This is the answer to chapter 2.4's problem. `GONZALES` and `GONZALEZ` no longer
fall into the same bucket as `GONZALES` and `MARTINEZ`: they land on a
high-similarity level with a substantial positive weight, while the unrelated
pair lands in "all other comparisons" with a negative one.

### Phonetic codes and term frequency

Two refinements, both of which matter on this data.

**Phonetic encoding.** Passing a Double Metaphone column to `NameComparison`
adds a level for names that *sound* alike without being spelt alike — the same
idea used for blocking in chapter 2.5, now used for scoring.

Note the two phonetic columns produced in the setup cell. Double Metaphone
returns *two* codes per name, a primary and an alternate. Blocking needs a single
comparable value, so `*_dm` holds the primary code as a string. Splink's
`dmeta_col_name` argument, by contrast, expects **both** codes as a list, which
is what `*_dmeta` holds. Passing the string form instead produces an opaque
database error (`No function matches ... list_distinct(VARCHAR)`) rather than a
helpful message.

**Term-frequency adjustments.** Chapter 2.4 noted that agreement on `GONZALEZ`
should be weaker evidence than agreement on a rare surname, and that the
framework as built could not express it. Term-frequency adjustment is the fix:
Splink counts how often each value occurs and scales the weight for that
specific value accordingly.

In [6]:
comparisons = [
    cl.NameComparison("nombre_clean", dmeta_col_name="nombre_dmeta")
        .configure(term_frequency_adjustments=True),
    cl.NameComparison("ap1_clean", dmeta_col_name="ap1_dmeta")
        .configure(term_frequency_adjustments=True),
    cl.NameComparison("ap2_clean", dmeta_col_name="ap2_dmeta")
        .configure(term_frequency_adjustments=True),
    cl.ExactMatch("sexo_clean"),
    cl.ExactMatch("nac_clean").configure(term_frequency_adjustments=True),
]

for c in comparisons:
    comp = c.get_comparison("duckdb")
    print(f"{comp.output_column_name:<14} {len(comp.comparison_levels)} levels")

nombre_clean   7 levels
ap1_clean      7 levels
ap2_clean      7 levels
sexo_clean     3 levels
nac_clean      3 levels


The name comparisons now have seven levels rather than six: the phonetic level
has been inserted below the loosest Jaro-Winkler level, catching names that sound
alike but are spelt too differently for a string-similarity measure to reach.

Sex gets a plain exact match: with two values there is nothing to be similar
about, and no term frequency worth adjusting for. Nationality gets term-frequency
adjustment because its two values are very unevenly distributed — agreeing on the
rarer one is more informative than agreeing on the common one.

## 4. Settings and the Linker

The settings object collects the three ingredients — link type, blocking rules,
comparisons — plus a few operational choices.

In [7]:
settings = SettingsCreator(
    link_type="link_only",
    blocking_rules_to_generate_predictions=blocking_rules,
    comparisons=comparisons,
    unique_id_column_name="unique_id",
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    input_table_or_tables=[fonasa_m, suseso_m],
    settings=settings,
    db_api=DuckDBAPI(),
    input_table_aliases=["fonasa", "suseso"],
)

print(type(linker))

<class 'splink.internals.linker.Linker'>


Two of those arguments are worth knowing about.

`link_type="link_only"` matches the specification from chapter 2.1: two clean
person-based files, no internal duplicates to resolve. The alternatives are
`dedupe_only` for one file and `link_and_dedupe` for both jobs at once.

`retain_intermediate_calculation_columns=True` keeps the per-level detail that
diagnostics need. Leave it out and `waterfall_chart()` later fails with a
`ValueError` — a common and confusing first encounter with Splink.

## 5. A first prediction, untrained

The model can already produce predictions. It should not be believed yet, and it
is worth seeing why.

In [8]:
df_predict = linker.inference.predict()
predictions = df_predict.as_pandas_dataframe()

print(f"candidate pairs scored: {len(predictions):,}")
print()
print("columns produced:")
print("  ", [c for c in predictions.columns if not c.startswith(("nombre_", "ap1_", "ap2_", "sexo_", "nac_"))])

Blocking time: 0.04 seconds
Predict time: 0.09 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'nombre_clean':
    m values not fully trained
Comparison: 'nombre_clean':
    u values not fully trained
Comparison: 'ap1_clean':
    m values not fully trained
Comparison: 'ap1_clean':
    u values not fully trained
Comparison: 'ap2_clean':
    m values not fully trained
Comparison: 'ap2_clean':
    u values not fully trained
Comparison: 'sexo_clean':
    m values not fully trained
Comparison: 'sexo_clean':
    u values not fully trained
Comparison: 'nac_clean':
    m values not fully trained
Comparison: 'nac_clean':
    u values not fully trained
The 'probability_two_random_records_match' setting has been set to the default value (0.0001). 
If this is not the desired beha

candidate pairs scored: 8,868

columns produced:
   ['match_weight', 'match_probability', 'source_dataset_l', 'source_dataset_r', 'unique_id_l', 'unique_id_r', 'gamma_nombre_clean', 'tf_nombre_clean_l', 'tf_nombre_clean_r', 'bf_nombre_clean', 'bf_tf_adj_nombre_clean', 'gamma_ap1_clean', 'tf_ap1_clean_l', 'tf_ap1_clean_r', 'bf_ap1_clean', 'bf_tf_adj_ap1_clean', 'gamma_ap2_clean', 'tf_ap2_clean_l', 'tf_ap2_clean_r', 'bf_ap2_clean', 'bf_tf_adj_ap2_clean', 'gamma_sexo_clean', 'bf_sexo_clean', 'gamma_nac_clean', 'tf_nac_clean_l', 'tf_nac_clean_r', 'bf_nac_clean', 'bf_tf_adj_nac_clean', 'match_key']


Splink emits a warning: the *m* values have not been estimated, so it is using
defaults. That warning is the point of this section.

Look at the `gamma_*` columns. Each records which comparison **level** a pair
landed on for that field: the highest number is an exact match, 0 is "all other
comparisons", −1 means the field could not be compared.

In [9]:
gamma_cols = [c for c in predictions.columns if c.startswith("gamma_")]

summary = pd.DataFrame({
    col: predictions[col].value_counts().sort_index()
    for col in gamma_cols
}).fillna(0).astype(int)
summary.index.name = "comparison level"
summary

,gamma_nombre_clean,gamma_ap1_clean,gamma_ap2_clean,gamma_sexo_clean,gamma_nac_clean
comparison level,,,,,
-1,494,0,129,357,478
0,5905,0,650,3308,2192
1,252,0,38,5203,6198
2,323,2181,1925,0,0
3,164,394,415,0,0
4,261,896,672,0,0
5,1469,5397,5039,0,0


This table is already informative, before any training.

Read the name columns. The highest level is an exact match; the levels below it
are the degrees of similarity. Thousands of pairs land on those **intermediate**
levels — similar but not identical. Every one of them was invisible to every
method in chapters 2.3 and 2.4, which could only see exact agreement. That is
where the recall improvement is going to come from.

One detail worth noticing, because it explains something in the next notebook:
`ap1_clean` has no pairs at all in its lowest levels. That is not a property of
the data, it is a property of the **blocking**: almost every rule requires the
first surname to agree exactly or phonetically, so a pair that disagrees on it
never became a candidate.

The candidate set is therefore not a random sample of pairs — it is heavily
conditioned by the blocking rules. This is precisely why *u* must be estimated
from *randomly drawn* pairs rather than from the candidates, and it is what
`estimate_u_using_random_sampling()` does in the next notebook.

In [10]:
print(predictions["match_probability"].describe().round(4).to_string())
print()
print("How many pairs would be accepted at a probability threshold of 0.9?")
print(f"  {(predictions['match_probability'] >= 0.9).sum():,}")

count    8868.0000
mean        0.3924
std         0.4494
min         0.0000
25%         0.0001
50%         0.0395
75%         0.9883
max         1.0000

How many pairs would be accepted at a probability threshold of 0.9?
  2,733


Do not read anything into those probabilities. They come from Splink's default
*m* values, not from your data — the model has been told the shape of the
question but not the answer to any part of it.

[The next notebook](nb07-comparisons-and-training.ipynb) estimates the three
parameters properly, compares the result with the values computed by hand in
chapter 2.4, and saves the trained model.